# 🔬 PaliGemma Model Evaluation for Assistive Navigation

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gómez Soteres  
**Universidad Francisco de Vitoria**

---

This notebook evaluates PaliGemma models for assistive navigation tasks.

## Metrics Evaluated:
- **Exact Match**: For closed yes/no questions
- **Keyword F1/Recall**: For element detection
- **CIDEr**: Standard image captioning metric
- **BERTScore**: Semantic similarity
- **METEOR**: Text generation quality
- **Inference Time**: Performance metric

## 1️⃣ Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0 accelerate bitsandbytes
!pip install -q torch torchvision
!pip install -q bert_score
!pip install -q pycocoevalcap
!pip install -q nltk
!pip install -q pandas numpy tqdm Pillow

# Download NLTK data for METEOR
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✅ All packages installed successfully!")

In [ ]:
# Mount Google Drive (for data only)
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted (for data access)!")

In [ ]:
# Setup project - Clone from GitHub (Private Repo)
import os
import sys
import subprocess

# ============================================================
# CONFIGURATION - PRIVATE REPO
# ============================================================
GITHUB_USER = "AlbertoGomezS"
GITHUB_REPO = "tfg-AlbertoGomez"
LOCAL_PROJECT_PATH = '/content/tfg'

# GitHub Personal Access Token (create at: https://github.com/settings/tokens)
# Required scopes: repo (full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # <-- REPLACE THIS

# ============================================================

# Clone or update from GitHub
if os.path.exists(LOCAL_PROJECT_PATH):
    # Update existing repo
    os.chdir(LOCAL_PROJECT_PATH)
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print("✅ Repository updated from GitHub")
    else:
        print(f"⚠️ Git pull issue: {result.stderr}")
else:
    # Clone with authentication (private repo)
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    result = subprocess.run(['git', 'clone', clone_url, LOCAL_PROJECT_PATH], capture_output=True, text=True)
    
    if result.returncode == 0:
        os.chdir(LOCAL_PROJECT_PATH)
        print(f"✅ Repository cloned from GitHub (private)")
    else:
        print(f"❌ Clone failed. Check your GITHUB_TOKEN")
        print(f"   Error: {result.stderr}")
        raise Exception("Could not clone repository - check token")

# Add src to path
sys.path.insert(0, LOCAL_PROJECT_PATH)

print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
# Import project modules
from src.model_loader import load_paligemma, unload_model, print_gpu_memory, get_model_info
from src.dataset_loader import (
    load_questions_config, 
    load_models_config,
    load_dataset, 
    create_evaluation_items,
    print_dataset_summary
)
from src.inference import (
    run_inference_on_dataset, 
    print_inference_summary,
    results_to_dict
)
from src.metrics import (
    calculate_all_metrics, 
    print_metrics_summary,
    metrics_to_dataframe
)

import json
import pandas as pd
from datetime import datetime

print("✅ All modules imported successfully!")

## 2️⃣ Configuration

In [ ]:
# ============================================================
# CONFIGURATION - PATHS
# ============================================================

# Data is stored in Google Drive (too large for Git)
DATA_PATH = '/content/drive/MyDrive/tfg/data'

# Config and src are cloned from GitHub
CONFIG_PATH = '/content/tfg/config'

# Results stored in Drive for persistence
RESULTS_PATH = '/content/drive/MyDrive/tfg/results'

# Create results directory
os.makedirs(RESULTS_PATH, exist_ok=True)

# Verify data path exists
if not os.path.exists(DATA_PATH):
    print(f"⚠️ Data path not found: {DATA_PATH}")
    print("   Please upload your data folder to Google Drive")
else:
    print(f"✅ Data path found: {DATA_PATH}")
    
print(f"📁 Config path: {CONFIG_PATH}")
print(f"📁 Results path: {RESULTS_PATH}")

In [ ]:
# Load configurations
questions_config = load_questions_config(f"{CONFIG_PATH}/questions.json")
models_config = load_models_config(f"{CONFIG_PATH}/models.json")

print(f"✅ Loaded {len(questions_config['categories'])} question categories")
print(f"✅ Loaded {len(models_config['models'])} model configurations")

In [ ]:
# ============================================================
# SELECT MODELS TO EVALUATE
# ============================================================

# For testing: Use only 1 model
TEST_MODE = True  # Set to False for full evaluation with all 13 models

if TEST_MODE:
    # Test with a single model first
    MODELS_TO_EVALUATE = [models_config['test_model']]
    print(f"🧪 TEST MODE: Evaluating 1 model: {MODELS_TO_EVALUATE[0]}")
else:
    # Evaluate all 13 models
    MODELS_TO_EVALUATE = [m['model_id'] for m in models_config['models']]
    print(f"🔬 FULL MODE: Evaluating {len(MODELS_TO_EVALUATE)} models")

print("\nModels to evaluate:")
for i, model in enumerate(MODELS_TO_EVALUATE, 1):
    print(f"  {i}. {model}")

## 3️⃣ Load Dataset

In [ ]:
# Load all images from the dataset
samples = load_dataset(DATA_PATH, questions_config, load_images=False)

# Print dataset summary
print_dataset_summary(samples)

In [ ]:
# Create evaluation items (image + question pairs)
eval_items = create_evaluation_items(samples, questions_config)

print(f"\n📊 Total evaluation items: {len(eval_items)}")
print(f"   (Each image has 2 questions)")

In [ ]:
# Preview some evaluation items
print("\n📋 Sample evaluation items:")
print("-" * 60)
for i, item in enumerate(eval_items[:6]):
    print(f"\n{i+1}. Image: {item.image_sample.image_id}")
    print(f"   Category: {item.image_sample.category}")
    print(f"   Question: {item.question.question_text}")
    print(f"   Type: {item.question.question_type}")
    if item.question.expected_answer:
        print(f"   Expected: {item.question.expected_answer}")

In [ ]:
# Login to HuggingFace (required for PaliGemma access)
from huggingface_hub import login

# IMPORTANT: Replace with your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
# Make sure you have accepted the PaliGemma license at: https://huggingface.co/google/paligemma-3b-pt-448

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"  # <-- REPLACE THIS

if HF_TOKEN != "YOUR_HUGGINGFACE_TOKEN":
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace")
else:
    print("⚠️ Please set your HuggingFace token above")
    print("   1. Get your token from: https://huggingface.co/settings/tokens")
    print("   2. Accept the PaliGemma license at: https://huggingface.co/google/paligemma-3b-pt-448")
    print("   3. Replace 'YOUR_HUGGINGFACE_TOKEN' with your actual token")

## 5️⃣ Run Evaluation

In [ ]:
# Store all results
all_model_results = {}
all_model_metrics = {}

print(f"🚀 Starting evaluation of {len(MODELS_TO_EVALUATE)} model(s)...")
print(f"📊 Total items to evaluate per model: {len(eval_items)}")
print("=" * 60)

In [ ]:
# Main evaluation loop
for model_idx, model_id in enumerate(MODELS_TO_EVALUATE, 1):
    print(f"\n{'='*60}")
    print(f"📦 Model {model_idx}/{len(MODELS_TO_EVALUATE)}: {model_id}")
    print(f"{'='*60}")
    
    # Get model info
    model_info = get_model_info(model_id)
    print(f"   Version: {model_info['version']}, Size: {model_info['size']}, "
          f"Type: {model_info['type']}, Resolution: {model_info['resolution']}")
    
    # Load model
    print("\n🔄 Loading model...")
    try:
        model, processor = load_paligemma(model_id)
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        continue
    
    # Run inference
    print("\n🔬 Running inference...")
    results = run_inference_on_dataset(
        model=model,
        processor=processor,
        eval_items=eval_items,
        max_new_tokens=100,
        show_progress=True
    )
    
    # Print inference summary
    print_inference_summary(results)
    
    # Convert results to dict format
    results_dict = results_to_dict(results)
    
    # Calculate metrics
    print("\n📊 Calculating metrics...")
    metrics = calculate_all_metrics(
        results_dict,
        calculate_cider_metric=True,
        calculate_bertscore_metric=True,
        calculate_meteor_metric=True
    )
    
    # Print metrics summary
    print_metrics_summary(metrics)
    
    # Store results
    all_model_results[model_id] = results_dict
    all_model_metrics[model_id] = metrics
    
    # Save results for this model
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_name_safe = model_id.replace("/", "_").replace("-", "_")
    
    # Save raw results
    results_file = f"{RESULTS_PATH}/{model_name_safe}_results_{timestamp}.json"
    with open(results_file, 'w') as f:
        json.dump(results_dict, f, indent=2)
    print(f"\n💾 Results saved to: {results_file}")
    
    # Save metrics
    metrics_file = f"{RESULTS_PATH}/{model_name_safe}_metrics_{timestamp}.json"
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"💾 Metrics saved to: {metrics_file}")
    
    # Unload model to free memory
    print("\n🧹 Unloading model...")
    unload_model(model, processor)
    print_gpu_memory()

print("\n" + "=" * 60)
print("✅ EVALUATION COMPLETE!")
print("=" * 60)

## 6️⃣ Results Analysis

In [ ]:
# Create comparison table
comparison_data = []

for model_id, metrics in all_model_metrics.items():
    overall = metrics.get('overall', {})
    model_info = get_model_info(model_id)
    
    row = {
        'Model': model_id.split('/')[-1],
        'Version': model_info['version'],
        'Size': model_info['size'],
        'Type': model_info['type'],
        'Resolution': model_info['resolution'],
        'Exact Match': round(overall.get('exact_match', 0), 4),
        'Keyword F1': round(overall.get('keyword_f1', 0), 4),
        'Keyword Recall': round(overall.get('keyword_recall', 0), 4),
        'CIDEr': round(overall.get('cider', 0), 4),
        'BERTScore F1': round(overall.get('bertscore_f1', 0), 4),
        'METEOR': round(overall.get('meteor', 0), 4),
        'Avg Time (s)': round(overall.get('avg_inference_time', 0), 2)
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print("\n📊 MODEL COMPARISON TABLE:")
print("=" * 100)
display(comparison_df)

In [ ]:
# Save comparison table to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
comparison_file = f"{RESULTS_PATH}/model_comparison_{timestamp}.csv"
comparison_df.to_csv(comparison_file, index=False)
print(f"💾 Comparison table saved to: {comparison_file}")

In [ ]:
# Show sample outputs from the evaluated model(s)
if all_model_results:
    last_model = list(all_model_results.keys())[-1]
    last_results = all_model_results[last_model]
    
    print(f"\n📋 Sample outputs from: {last_model}")
    print("=" * 80)
    
    # Show 10 random samples
    import random
    sample_size = min(10, len(last_results))
    samples = random.sample(last_results, sample_size)
    
    for i, result in enumerate(samples, 1):
        print(f"\n--- Sample {i} ---")
        print(f"Image: {result['image_id']}")
        print(f"Category: {result['category']}")
        print(f"Question: {result['question_text']}")
        print(f"Model Response: {result['model_response']}")
        if result.get('expected_answer'):
            print(f"Expected: {result['expected_answer']}")
        print(f"Keywords Expected: {result.get('expected_keywords', [])}")

## 7️⃣ Save Results to Google Drive

In [ ]:
# Copy results to Google Drive for persistence
DRIVE_RESULTS_PATH = '/content/drive/MyDrive/tfg/results'
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)

# Copy all result files
for file in os.listdir(RESULTS_PATH):
    src = os.path.join(RESULTS_PATH, file)
    dst = os.path.join(DRIVE_RESULTS_PATH, file)
    shutil.copy2(src, dst)

print(f"✅ All results copied to Google Drive: {DRIVE_RESULTS_PATH}")
print(f"\nFiles saved:")
for file in os.listdir(DRIVE_RESULTS_PATH):
    print(f"  - {file}")

---

## ✅ Evaluation Complete!

### Results have been saved to:
- **Local**: `/content/tfg/results/`
- **Drive**: `/content/drive/MyDrive/tfg/results/`

### Files generated:
- `*_results_*.json` - Raw inference results
- `*_metrics_*.json` - Calculated metrics  
- `model_comparison_*.csv` - Comparison table

### Next Steps:
1. Download results from Google Drive
2. Analyze metrics in detail
3. Create visualizations for TFG memoria
4. If TEST_MODE was True, set it to False and run full evaluation with all 13 models